In [46]:
import pandas as pd

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 39073 entries, 0 to 39072
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   sampleid             39073 non-null  int64
 1   age                  39073 non-null  int64
 2   workclass            36849 non-null  str  
 3   fnlwgt               39073 non-null  int64
 4   education            39073 non-null  str  
 5   educational_num      39073 non-null  int64
 6   marital_status       39073 non-null  str  
 7   occupation           36839 non-null  str  
 8   relationship         39073 non-null  str  
 9   race                 39073 non-null  str  
 10  gender               39073 non-null  str  
 11  capital_gain         39073 non-null  int64
 12  capital_loss         39073 non-null  int64
 13  hours_per_week       39073 non-null  int64
 14  native_country       38392 non-null  str  
 15  income               39073 non-null  str  
 16  profile_description  39073 non-nu

In [47]:
train['income'].unique()

<StringArray>
['<=50K', '>50K']
Length: 2, dtype: str

In [48]:
train['income'] = (train['income'] == '>50K').astype(int)
train['income'].sum()

np.int64(9349)

In [49]:
no_us = train[train['native_country'] != 'United-States'].dropna(subset=['native_country'])
counts = no_us.groupby('native_country')['income'].sum()
answer_1 = counts.idxmax()
print(isinstance(answer_1, str))

True


In [50]:
rates = train.groupby('occupation')['income'].mean()
answer_2 = rates.idxmax()

In [ ]:
import numpy as np

target = 'income'
drop_cols = ['sampleid', 'education', 'profile_description']

numeric_cols = ['age', 'fnlwgt', 'educational_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical_cols = ['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'gender', 'native_country']

X = train[numeric_cols + categorical_cols]
y = train[target]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_steps = Pipeline([
	('impute', SimpleImputer(strategy='median')),
	('scale', StandardScaler()),
])

categorical_steps = Pipeline([
	('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
	('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer([
	('num', numeric_steps, numeric_cols),
	('cat', categorical_steps, categorical_cols),
])

model = Pipeline([
	('pre', preprocess),
	('model', LogisticRegression(max_iter=2000))
])

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
oof = cross_val_predict(model, X, y, cv=skf, method='predict_proba')[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y, oof >= t, average='macro') for t in thresholds]
best = int(np.argmax(f1s))

33


In [56]:
print(round(f1s[best], 4), thresholds[best])

0.7922 0.37999999999999995


In [55]:
threshold = thresholds[best]
model.fit(X, y)

X_test = test[numeric_cols + categorical_cols]
proba_test = model.predict_proba(X_test)[:, 1]
predict_test = (proba_test >= threshold).astype(int)

print(predict_test.mean())

0.24966731497594433


In [ ]:
from lightgbm import LGBMClassifier

def prepare_lgbm(df, reference=None):
	out = df[numeric_cols + categorical_cols].copy()
	for col in categorical_cols:
		values = out[col].fillna('Unknown')
		if reference is None:
			out[col] = values.astype('category')
		else:
			out[col] = pd.Categorical(values, categories=reference[col].cat.categories)
	return out

X_lgb = prepare_lgbm(train)
X_test_lgb = prepare_lgbm(test, reference=X_lgb)

/tmp/ipykernel_2225258/3568542766.py:10: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  out[col] = pd.Categorical(values, categories=reference[col].cat.categories)


In [ ]:
lgb_model = LGBMClassifier(
	n_estimators=300,
	learning_rate=0.05,
	random_state=67,
	verbose=-1
)

In [62]:
oof_lgb = cross_val_predict(lgb_model, X_lgb, y, cv=skf, method='predict_proba')[:, 1]
f1s_lgb = [f1_score(y, oof_lgb >= t, average='macro') for t in thresholds]
best_lgb = int(np.argmax(f1s_lgb))
print("Best score for LightGBM is ", round(f1s_lgb[best_lgb], 4), "at threshold ", thresholds[best_lgb])
print('Best score for Logistic Regression is ', round(f1s[best], 4), "at threshold ", thresholds[best])

Best score for LightGBM is  0.8201 at threshold  0.42999999999999994
Best score for Logistic Regression is  0.7922 at threshold  0.37999999999999995


In [ ]:
# clustering by profile_description
for text in train['profile_description'].sample(n=10, random_state=67):
	print(text, '\n')

neural encryption scalable development assessment backend 

margin pitch dividend monitoring training experience 

transparency amendment monitoring regulation statute standards 

timber assessment forklift rebar management blueprint torque support 

quality tableware operations linen courtesy sanitization amenity 

KPI support procedures prospectus liquidity certified 

coordination etiquette management catering wardrobe certified garnish 

quality workshop masonry management monitoring excavation 

hydraulic procedures wiring masonry scaffold operations timber 

upholstery grooming professional linen assessment reservation compliance garnish 



In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer

text = pd.concat([train['profile_description'], test['profile_description']]).fillna('')
n_train = len(train)

tfidf = TfidfVectorizer(stop_words='english', min_df=5, max_df=0.5, sublinear_tf=True)

X_text = tfidf.fit_transform(text)
vocab = tfidf.get_feature_names_out()
print(X_test.shape)

(9769, 14)


In [75]:
order = np.argsort(tfidf.idf_)
print(vocab[order[:20]], '\n\n')
print(vocab[order[-20:]])

['analysis' 'management' 'professional' 'experience' 'reporting'
 'development' 'training' 'compliance' 'maintenance' 'standards'
 'certified' 'procedures' 'operations' 'monitoring' 'assessment'
 'coordination' 'planning' 'support' 'inspection' 'quality'] 


['scalable' 'bandwidth' 'runtime' 'cloud' 'neural' 'pipeline'
 'microservice' 'database' 'automated' 'algorithm' 'repository' 'compiler'
 'server' 'syntax' 'framework' 'debug' 'containerized' 'encryption'
 'backend' 'deploy']


In [ ]:
print(X_text.shape)

doc_freq = (X_text > 0).sum(axis=0).A1       # how many descriptions contain each word
vocab_df = (pd.DataFrame({'word': vocab, 'docs': doc_freq})
			  .sort_values('docs', ascending=False))
pd.set_option('display.max_rows', 500)
print(vocab_df)

(48842, 138)
              word  docs
5         analysis  6240
71      management  6156
92    professional  6156
101      reporting  6155
45      experience  6155
35     development  6148
127       training  6141
24      compliance  6131
70     maintenance  6115
117      standards  6103
19       certified  6092
90      procedures  6090
81      operations  6086
77      monitoring  6080
6       assessment  6064
29    coordination  6053
87        planning  6042
121        support  6036
59      inspection  6020
95         quality  5964
136         wiring  2420
124         timber  2415
60      insulation  2402
88        plumbing  2398
137       workshop  2395
58       hydraulic  2394
44      excavation  2391
38         drywall  2381
64           lathe  2380
13       blueprint  2364
49        forklift  2353
26        concrete  2351
126         torque  2344
54         grinder  2343
135        welding  2343
125        toolbox  2339
105        rigging  2334
109       scaffold  2328
31          

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

filler = vocab_df.loc[vocab_df['docs'] > 5000, 'word'].tolist()
print(len(filler), filler)

tfidf = TfidfVectorizer(stop_words=filler, min_df=5, sublinear_tf=True)
X_text = tfidf.fit_transform(text)
vocab = tfidf.get_feature_names_out()
print(X_text.shape)

k = train['occupation'].nunique()
km = KMeans(n_clusters=k, n_init=10, random_state=67)
labels = km.fit_predict(X_text)

for c in range(k):
	top = km.cluster_centers_[c].argsort()[::-1][:8]
	print(f'{c:2d}  size={np.sum(labels == c):5d}  ' + ', '.join(vocab[top]))

train_labels = labels[:n_train]
for col in ['occupation', 'workclass', 'educational_num', 'relationship']:
	ari = adjusted_rand_score(train[col].fillna('Unknown'), train_labels)
	print(f'{col:16s} ARI = {ari:.3f}')

20 ['analysis', 'management', 'professional', 'reporting', 'experience', 'development', 'training', 'compliance', 'maintenance', 'standards', 'certified', 'procedures', 'operations', 'monitoring', 'assessment', 'coordination', 'planning', 'support', 'inspection', 'quality']
(48842, 118)
 0  size= 2871  revenue, strategy, dividend, leverage, branding, acquisition, ipo, valuation
 1  size= 3942  forklift, hydraulic, scaffold, plumbing, blueprint, timber, grinder, excavation
 2  size= 4166  wiring, timber, excavation, carpentry, rigging, concrete, welding, plumbing
 3  size= 4594  merger, margin, kpi, shareholder, stakeholder, equity, valuation, pitch
 4  size= 2363  insulation, timber, toolbox, ductwork, grinder, drywall, excavation, welding
 5  size= 4188  tableware, sanitization, etiquette, linen, concierge, beverage, reception, spa
 6  size= 4204  software, agile, frontend, refactor, scalable, bandwidth, runtime, cloud
 7  size= 4530  municipal, enforcement, subsidy, transparency, cle

In [ ]:
occ = train['occupation'].fillna('Unknown')

for k in range(3, 16):
	lab = KMeans(n_clusters=k, n_init=10, random_state=67).fit_predict(X_text)
	print(f'k={k:2d}  ARI vs occupation = {adjusted_rand_score(occ, lab[:n_train]):.3f}')

k= 3  ARI vs occupation = 0.253
k= 4  ARI vs occupation = 0.357
k= 5  ARI vs occupation = 0.361
k= 6  ARI vs occupation = 0.351
k= 7  ARI vs occupation = 0.343
k= 8  ARI vs occupation = 0.280
k= 9  ARI vs occupation = 0.335
k=10  ARI vs occupation = 0.263
k=11  ARI vs occupation = 0.262
k=12  ARI vs occupation = 0.254
k=13  ARI vs occupation = 0.239
k=14  ARI vs occupation = 0.232
k=15  ARI vs occupation = 0.225


In [ ]:
km5 = KMeans(n_clusters=5, n_init=10, random_state=67)
labels5 = km5.fit_predict(X_text)

for c in range(5):
	top = km5.cluster_centers_[c].argsort()[::-1][:8]
	print(f'{c}  size={np.sum(labels5 == c):5d}  ' + ', '.join(vocab[top]))

print(pd.crosstab(occ, labels5[:n_train], normalize='index').round(2))

0  size=10651  equity, valuation, merger, acquisition, kpi, strategy, liquidity, pitch
1  size=14141  wiring, timber, workshop, insulation, plumbing, hydraulic, excavation, lathe
2  size= 4204  software, agile, frontend, refactor, scalable, bandwidth, runtime, cloud
3  size=12986  spa, beverage, garnish, tableware, catering, reception, pantry, banquet
4  size= 6860  municipal, enforcement, legislation, transparency, civic, amendment, regulation, dispatch
col_0                 0     1     2     3     4
occupation                                     
Adm-clerical       0.00  0.00  0.00  0.77  0.23
Armed-Forces       0.00  0.00  0.00  0.00  1.00
Craft-repair       0.00  0.94  0.00  0.00  0.06
Exec-managerial    0.86  0.00  0.00  0.00  0.14
Farming-fishing    0.00  0.95  0.00  0.00  0.05
Handlers-cleaners  0.00  0.94  0.00  0.00  0.06
Machine-op-inspct  0.00  0.98  0.00  0.00  0.02
Other-service      0.00  0.00  0.00  0.89  0.11
Priv-house-serv    0.00  0.00  0.00  1.00  0.00
Prof-specialt

In [84]:
wc = train['workclass'].fillna('Unknown')
print(pd.crosstab(wc, labels5[:n_train], normalize='index').round(2))

col_0                0     1     2     3     4
workclass                                     
Federal-gov       0.00  0.00  0.00  0.00  1.00
Local-gov         0.00  0.00  0.00  0.00  1.00
Never-worked      0.00  0.00  0.00  1.00  0.00
Private           0.25  0.36  0.10  0.28  0.01
Self-emp-inc      0.60  0.19  0.13  0.07  0.00
Self-emp-not-inc  0.30  0.44  0.13  0.12  0.00
State-gov         0.00  0.00  0.00  0.00  1.00
Unknown           0.00  0.00  0.00  1.00  0.00
Without-pay       0.00  0.81  0.00  0.19  0.00


In [85]:
sector_of = pd.crosstab(occ, labels5[:n_train]).idxmax(axis=1)   # occupation -> its main cluster
is_gov = wc.str.contains('gov')

rule = occ.map(sector_of).where(~is_gov, 4)    # government workers -> cluster 4, everyone else by occupation
print('ARI vs rule =', round(adjusted_rand_score(rule, labels5[:n_train]), 3))

ARI vs rule = 0.954


In [88]:
km5 = KMeans(n_clusters=5, n_init=50, random_state=67)
labels5 = km5.fit_predict(X_text)
print(km5.inertia_, round(adjusted_rand_score(rule, labels5[:n_train]), 3))

40685.287332821375 0.954


In [89]:
threshold_lgb = thresholds[best_lgb]
lgb_model.fit(X_lgb, y)

proba_test_lgb = lgb_model.predict_proba(X_test_lgb)[:, 1]
pred_income = np.where(proba_test_lgb >= threshold_lgb, '>50K', '<=50K')
print('share >50K:', round((pred_income == '>50K').mean(), 4))

test_clusters = labels5[n_train:]
print(len(test_clusters) == len(test))
print(pd.Series(test_clusters).value_counts(normalize=True).sort_index().round(3))

share >50K: 0.2272
True
0    0.213
1    0.292
2    0.088
3    0.271
4    0.136
Name: proportion, dtype: float64


In [90]:
single = pd.DataFrame([
	{'subtaskID': 1, 'datapointID': 1, 'answer': answer_1},
	{'subtaskID': 2, 'datapointID': 1, 'answer': answer_2},
])

sub3 = pd.DataFrame({
	'subtaskID': 3,
	'datapointID': test['sampleid'],
	'answer': pred_income,
})

sub4 = pd.DataFrame({
	'subtaskID': 4,
	'datapointID': test['sampleid'],
	'answer': test_clusters,
})

submission = pd.concat([single, sub3, sub4], ignore_index=True)
submission.to_csv('submission.csv', index=False)
print(submission.groupby('subtaskID').size())

subtaskID
1       1
2       1
3    9769
4    9769
dtype: int64
